# ARC3 27B offline CPU preflight

Checks actual target/draft mounts and runtime source. No model inference or game access. Upstream Qwen, Z-Lab and Pennyroyal attribution retained.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json,zipfile,subprocess,time
started=time.monotonic()
def locate(owner,slug,framework,variation):
    roots=[Path('/kaggle/input/models')/owner/slug/framework/variation/'1',
           Path('/kaggle/input')/slug/framework/variation/'1',
           Path('/kaggle/input')/owner/slug/framework/variation/'1']
    hits=[p for p in roots if (p/'config.json').is_file()]
    if len(hits)!=1:raise ValueError('Expected exact model mount: '+str(roots))
    return hits[0]
target=locate('mikedan7','qwen3-8-27b-fp8-official','pytorch','hf-fp8')
draft=locate('dangkhoa2016','z-lab-qwen3-8-27b-dflash2','transformers','default')
tc=json.loads((target/'config.json').read_text());dc=json.loads((draft/'config.json').read_text())
assert tc['architectures']==['Qwen3_5ForConditionalGeneration'],tc.get('architectures')
q=tc.get('quantization_config',{})
assert q.get('quant_method')=='fp8' and q.get('weight_block_size')==[128,128],q
index=json.loads((target/'model.safetensors.index.json').read_text())
files={}
for name in sorted(set(index['weight_map'].values())):
    path=(target/name).resolve()
    if not path.is_relative_to(target.resolve()) or not path.is_file():raise ValueError('Invalid indexed shard')
    files[name]=path.stat().st_size
assert (draft/'model.safetensors').is_file()
roots=[Path('/kaggle/input/datasets/dfranzen/pennyroyal-v253'),Path('/kaggle/input/pennyroyal-v253')]
wheelroot=next(p for p in roots if p.is_dir())
wheels=list(wheelroot.rglob('sglang-*.whl'));assert len(wheels)==1
with zipfile.ZipFile(wheels[0]) as archive:
    names=archive.namelist()
    server=next(n for n in names if n.endswith('/srt/server_args.py'))
    source=archive.read(server).decode()
    # Some arguments are generated by imported dataclass configurations rather
    # than literal strings in server_args.py. Inspect all actual runtime sources.
    all_source='\n'.join(archive.read(n).decode(errors='replace') for n in names if n.endswith('.py'))
    flags={flag:(flag in all_source or flag.lstrip('-').replace('-','_') in all_source)
           for flag in ['DFLASH','--speculative-draft-window-size','--speculative-attention-mode','--decode-attention-backend','--max-mamba-cache-size']}
    assert all(flags.values()),flags
    implementations=[n for n in names if 'dflash' in n.lower() and n.endswith('.py')]
    assert implementations,'No DFlash implementation in actual serving wheel'
metadata={}
for label,root in [('target',target),('draft',draft)]:
    metadata[label]={}
    for name in ['config.json','tokenizer.json','tokenizer_config.json','chat_template.jinja','preprocessor_config.json','processor_config.json']:
        p=root/name
        if p.is_file():metadata[label][name]={'sha256':sha256(p.read_bytes()).hexdigest(),'size':p.stat().st_size}
device=subprocess.run(['sh','-c','command -v nvidia-smi >/dev/null && nvidia-smi -L || true'],text=True,capture_output=True)
report=dict(passed=True,elapsed_seconds=time.monotonic()-started,target_path=str(target),draft_path=str(draft),
    target_config=tc,draft_config=dc,metadata=metadata,target_weight_files=files,
    draft_weight_bytes=(draft/'model.safetensors').stat().st_size,
    wheel_name=wheels[0].name,wheel_sha256=sha256(wheels[0].read_bytes()).hexdigest(),
    runtime_flags=flags,runtime_implementations=implementations,device_census=device.stdout.strip(),
    boundary='CPU mount/config/file-existence and serving-source inspection only. Option strings or configuration fields found in source do not certify a successful CLI launch. No model loading, weight correctness, draft compatibility, GPU inference, training support or gameplay quality established.')
Path('/kaggle/working/qwen27-preflight.json').write_text(json.dumps(report,indent=2)+'\n')
print(json.dumps({k:v for k,v in report.items() if k not in ['target_config','draft_config','metadata','target_weight_files']},indent=2))
